# Challenge 6: ReadyNow Emergency Preparedness Assistant

This notebook implements the Federal Emergency Machine Assistant (ReadyNow) case study using Google ADK.

## What this notebook demonstrates
- Root coordinator agent with weather, search, route, and Q&A workflow specialists
- Sequential answer refinement (`qa_agent -> critique_agent -> refine_agent`)
- Callback logging and user input validation
- Local testing with streamed agent events
- Deployment to Vertex AI Agent Platform (Agent Engine)
- Remote post-deployment test

Run each cell in order in Colab Enterprise.

In [ ]:
# Step 1: Install dependencies (restart runtime once after first install if needed)
%pip install -q -r requirements.txt

In [ ]:
# Step 2: Runtime configuration
import logging
import os

import vertexai

from agent.config import load_runtime_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "TRUE")

cfg = load_runtime_config()
assert cfg.project_id, "Set GOOGLE_CLOUD_PROJECT before continuing."
assert cfg.maps_api_key, "Set GOOGLE_MAPS_API_KEY before continuing."

os.environ["GOOGLE_CLOUD_PROJECT"] = cfg.project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = cfg.location
os.environ["GEMINI_MODEL"] = cfg.model

vertexai.init(project=cfg.project_id, location=cfg.location)
print(f"Configured project={cfg.project_id}, location={cfg.location}, model={cfg.model}")

In [ ]:
# Step 3: Build the ReadyNow root agent
from agent.root_agent import build_ready_now_root_agent
from lib.local_runner import list_authors, make_local_app, stream_local_query

root_agent = build_ready_now_root_agent(model=cfg.model)
app = make_local_app(root_agent, enable_tracing=False)

print(f"Built root agent: {root_agent.name}")
print("Sub-agents:")
for sub in root_agent.sub_agents:
    print("-", sub.name)

In [ ]:
# Step 4: Local test prompts with streamed-event evidence
prompts = [
    "I am in Miami, FL. Summarize the weather risk and immediate safety actions.",
    "Find current FEMA guidance for flood preparedness in Texas.",
    "Give me an evacuation route from Santa Monica, CA to Pasadena, CA.",
    "What should a family in a wildfire-prone area do in the next 24 hours to prepare?",
]

for idx, message in enumerate(prompts, start=1):
    final_text, events = stream_local_query(app, message=message, user_id=f"local-test-{idx}")
    print("=" * 80)
    print(f"Prompt {idx}: {message}")
    print("Authors:", list_authors(events))
    print("Final response:\n", final_text[:1500])

## Architecture diagram prompt for image models

Use this text prompt with your preferred image model:

> Create a clean technical architecture diagram titled "FEMA ReadyNow Emergency Preparedness Multi-Agent System". Show: (1) User chat interface, (2) Root coordinator agent, (3) Weather Forecast Agent calling Google Weather API and Geocoding, (4) Internet Search Agent calling Google Search, (5) Route Agent calling Google Maps Routes API, (6) Q&A Agent for safety responses, (7) Sequential validation pipeline with Answer -> Critique -> Refine stages, (8) Callback layer for prompt/response logging and user-input validation, (9) Agent Platform deployment boundary on Google Cloud, (10) Frontend service on Cloud Run invoking deployed Agent Engine. Include directional arrows for data flow, grouped boxes for Local Testing vs Deployed Runtime, and labels for session state and event streaming. Style: modern, minimal, high contrast, white background, blue/gray palette, legible text, presentation-ready.

In [ ]:
# Step 5: Deploy to Agent Platform
from google.cloud import storage

from lib.remote_client import deploy_agent


def ensure_staging_bucket(project_id: str, bucket_name: str, location: str) -> str:
    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)
    if bucket.exists():
        return f"gs://{bucket_name}"

    bucket = client.create_bucket(bucket_name, location=location)
    return f"gs://{bucket.name}"


STAGING_BUCKET_NAME = f"{cfg.project_id}-agent-engine-staging"
STAGING_BUCKET_URI = ensure_staging_bucket(cfg.project_id, STAGING_BUCKET_NAME, cfg.location)
print("Using staging bucket:", STAGING_BUCKET_URI)

RUN_DEPLOY = False  # Set to True when ready to deploy.
remote_agent = None
if RUN_DEPLOY:
    remote_agent = deploy_agent(
        root_agent=root_agent,
        project_id=cfg.project_id,
        location=cfg.location,
        staging_bucket=STAGING_BUCKET_URI,
        display_name="readynow-emergency-assistant",
    )
    print("Deployment name:", getattr(remote_agent, "name", remote_agent))
else:
    print("Deployment skipped. Set RUN_DEPLOY=True to deploy.")

In [ ]:
# Step 6: Test a deployed agent by resource name
from lib.remote_client import get_remote_agent, stream_remote_query

AGENT_ENGINE_RESOURCE_NAME = os.getenv("AGENT_ENGINE_RESOURCE_NAME", "").strip()
if not AGENT_ENGINE_RESOURCE_NAME:
    print("Set AGENT_ENGINE_RESOURCE_NAME to run remote verification.")
else:
    deployed = get_remote_agent(
        resource_name=AGENT_ENGINE_RESOURCE_NAME,
        project_id=cfg.project_id,
        location=cfg.location,
    )
    remote_answer, remote_events = stream_remote_query(
        remote_agent=deployed,
        user_id="challenge-6-remote-test",
        message="I am in Sacramento, CA. What should I do right now if wildfire smoke worsens?",
    )
    print("Remote authors:", [e.get("author") for e in remote_events if e.get("author")][:20])
    print("Remote answer:\n", remote_answer[:2000])

## Step 7: Run deployed integration tests

From a shell in `challenge-6/`:

```bash
python -m pip install -r requirements-dev.txt
export GOOGLE_CLOUD_PROJECT="your-project-id"
export GOOGLE_CLOUD_LOCATION="us-central1"
export AGENT_ENGINE_RESOURCE_NAME="projects/.../locations/.../reasoningEngines/..."
pytest -m integration -q
```

The tests are in `tests/test_deployed_integration.py` and are designed to validate both successful response generation and malicious-input blocking.